In [6]:
import pandas as pd

data = pd.read_csv("data/ChargePointData2011-2020-ManullyPreCleaned3.csv", low_memory=False)
data

,Station Name,MAC Address,Org Name,Start Date,Start Time Zone,End Date,End Time Zone,Transaction Date (Pacific Time),Total Duration (hh:mm:ss),Charging Time (hh:mm:ss),...,Longitude,Currency,Fee,Ended By,Plug In Event Id,Driver Postal Code,User ID,County,System S/N,Model Number
0,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/29/2011 20:17,PDT,7/29/2011 23:20,PDT,7/29/2011 23:20,03:03:32,01:54:03,...,-122.160309,USD,0.00,Plug Out at Vehicle,3,95124.0,3284,NaN,NaN,NaN
1,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 0:00,PDT,7/30/2011 0:02,PDT,7/30/2011 0:02,00:02:06,00:01:54,...,-122.160309,USD,0.00,Customer,4,94301.0,4169,NaN,NaN,NaN
2,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 8:16,PDT,7/30/2011 12:34,PDT,7/30/2011 12:34,04:17:32,04:17:28,...,-122.160309,USD,0.00,Plug Out at Vehicle,5,94301.0,4169,NaN,NaN,NaN
3,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 14:51,PDT,7/30/2011 16:55,PDT,7/30/2011 16:55,02:03:24,02:02:58,...,-122.160309,USD,0.00,Customer,6,94302.0,2545,NaN,NaN,NaN
4,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 18:51,PDT,7/30/2011 20:03,PDT,7/30/2011 20:03,01:11:24,00:43:54,...,-122.160309,USD,0.00,Plug Out at Vehicle,7,94043.0,3765,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259410,PALO ALTO CA / TED THOMPSON #3,0024:B100:0002:9F81,City of Palo Alto,12/31/2020 16:39,PST,12/31/2020 17:14,PST,12/31/2020 17:16,00:35:44,00:35:18,...,-122.144043,USD,0.42,Plug Out at Vehicle,2369,94301.0,2024201,NaN,1.750000e+11,CT4020-HD-GW
259411,PALO ALTO CA / WEBSTER #1,0024:B100:0002:4233,City of Palo Alto,12/31/2020 16:48,PST,12/31/2020 20:31,PST,12/31/2020 20:32,03:43:01,02:14:04,...,-122.158272,USD,4.69,Plug Out at Vehicle,7255,94541.0,242661,Santa Clara County,1.340000e+11,CT4020-HD-GW
259412,PALO ALTO CA / MPL #4,0024:B100:0003:CD37,City of Palo Alto,12/31/2020 17:28,PST,1/1/2021 10:56,PST,1/1/2021 10:57,17:28:14,05:08:12,...,-122.113510,USD,6.82,Plug Out at Vehicle,455,94306.0,449995,Santa Clara County,2.000000e+11,CT4010-HD-GW
259413,PALO ALTO CA / WEBSTER #1,0024:B100:0002:4233,City of Palo Alto,12/31/2020 17:49,PST,12/31/2020 21:55,PST,12/31/2020 21:56,04:05:34,03:53:42,...,-122.158272,USD,2.75,Plug Out at Vehicle,7256,94552.0,804075,Santa Clara County,1.340000e+11,CT4020-HD-GW


In [7]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 259415 entries, 0 to 259414
Data columns (total 33 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   Station Name                     259415 non-null  str    
 1   MAC Address                      259415 non-null  str    
 2   Org Name                         259415 non-null  str    
 3   Start Date                       259415 non-null  str    
 4   Start Time Zone                  259415 non-null  str    
 5   End Date                         259415 non-null  str    
 6   End Time Zone                    259415 non-null  str    
 7   Transaction Date (Pacific Time)  259207 non-null  str    
 8   Total Duration (hh:mm:ss)        259415 non-null  str    
 9   Charging Time (hh:mm:ss)         259415 non-null  str    
 10  Energy (kWh)                     259415 non-null  float64
 11  GHG Savings (kg)                 259415 non-null  float64
 12  Gasoline Savi

In [ ]:
import numpy as np


def duration_to_hours(series: pd.Series) -> pd.Series:
    """Parse hh:mm:ss duration strings into hours."""
    parts = series.str.split(":", expand=True).astype(float)
    return (parts[0] * 3600 + parts[1] * 60 + parts[2]) / 3600


def assign_tier(price) -> np.ndarray:
    """Map $/kWh price to tier: 0 -> 0, <15 -> 1, <25 -> 2, else 3."""
    arr = np.asarray(price, dtype=float)
    tier = np.select(
        [arr == 0, arr < 15, arr < 25],
        [0, 1, 2],
        default=3,
    )
    return np.where(np.isnan(arr), np.nan, tier)


# Charging duration (hours) and energy delivered per session
data["charging_duration_hours"] = duration_to_hours(data["Charging Time (hh:mm:ss)"])
data["energy_kwh"] = data["Energy (kWh)"]

# Raw effective $/kWh: total session fee divided by energy (captures idle/time surcharges)
data["raw_price_per_kwh"] = np.where(
    data["energy_kwh"] > 0,
    data["Fee"] / data["energy_kwh"],
    np.nan,
)

# Median unit price per station (robust to idle-fee outliers vs mean)
unit_price_by_station = (
    data.loc[data["raw_price_per_kwh"].notna()]
    .groupby("Station Name", as_index=False)
    .agg(
        median_unit_price_per_kwh=("raw_price_per_kwh", "median"),
        session_count=("raw_price_per_kwh", "count"),
    )
    .sort_values("median_unit_price_per_kwh", ascending=False)
)
unit_price_by_station["price_tier"] = assign_tier(
    unit_price_by_station["median_unit_price_per_kwh"]
)

data["price_tier"] = assign_tier(data["raw_price_per_kwh"])

unit_price_by_station

TypeError: data type 'Int64' not understood

In [ ]:
# Attach station-level median unit price back to each session
data = data.merge(
    unit_price_by_station[
        ["Station Name", "median_unit_price_per_kwh", "price_tier"]
    ].rename(columns={"price_tier": "station_price_tier"}),
    on="Station Name",
    how="left",
)

data[
    [
        "Station Name",
        "Fee",
        "energy_kwh",
        "charging_duration_hours",
        "raw_price_per_kwh",
        "price_tier",
        "median_unit_price_per_kwh",
        "station_price_tier",
    ]
].query("Fee > 0").head(10)

,Station Name,Fee,energy_kwh,charging_duration_hours,raw_price_per_kwh,median_unit_price_per_kwh
132452,PALO ALTO CA / CAMBRIDGE #2,1.21,5.273,1.665278,0.229471,0.000000
132453,PALO ALTO CA / HAMILTON #2,2.36,10.239,3.083056,0.230491,0.000000
132454,PALO ALTO CA / WEBSTER #1,1.84,8.010,2.630278,0.229713,0.228724
132455,PALO ALTO CA / WEBSTER #1,2.70,11.730,1.950000,0.230179,0.228724
132456,PALO ALTO CA / HAMILTON #1,1.80,7.843,1.484722,0.229504,0.000000
132457,PALO ALTO CA / HAMILTON #2,0.16,0.673,0.446667,0.237741,0.000000
132458,PALO ALTO CA / HIGH #3,2.95,12.828,2.278056,0.229966,0.229885
132459,PALO ALTO CA / CAMBRIDGE #2,2.58,11.220,3.726944,0.229947,0.000000
132460,PALO ALTO CA / RINCONADA LIB 1,0.72,3.131,0.535833,0.229958,0.000000
132461,PALO ALTO CA / MPL #5,2.59,11.250,1.916944,0.230222,0.000000
